In [0]:
import re

import data.utilities.common as Commonlib

import pyspark.sql.functions as F
from beertech_utils.core.logger import LogManager

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_entries = Commonlib.list_files(CONFIG_BASE_PATH, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

In [0]:
logger = LogManager.get_logger("global_package_xref_procedure")

try:
    config = Commonlib.get_object_config(config_file_path)

    # Instantiate medallion object
    medallion = Medallion(config)
except Exception as e:
    logger.error("Error creating medallion object for global_package_xref")
    raise

In [0]:
SOURCE_TABLES = {
    # silver source tables
    "global_us_pkg_xref": MTU.get_fully_qualified_table_name("oracle", "whsusr", "vw_global_us_pkg_xref"),
}

In [0]:
# Columns Mapping

col_map = {
    "glbl_pkg_cd": F.substring(F.col("global_pack_nm"), 1, 2),
    "glbl_pkg_nm": F.regexp_extract(F.col("global_pack_nm"), r"^\d{1,2}:\s*(.*)$", 1),
    "glbl_subpkg_cd": F.col("global_sub_pack_id"),
    "glbl_subpkg_nm": F.col("global_sub_pack_nm"),
    "us_pkg_cd": F.col("us_pack_id"),
    "us_subpkg_cd": F.col("us_sub_pack_id"),
    "__created_tsp": F.lit(medallion.start_datetime).cast(TimestampType()),
}

In [0]:
try:
    # Extracts primary keys from config
    pks = list(
        filter(
            None, map(lambda x: x.get("name") if x.get("metadata").get("key") == True else "", config["gold"]["schema"])
        )
    )

    wind = Window.partitionBy("glbl_subpkg_cd", "glbl_pkg_cd").orderBy(F.col("us_subpkg_cd").desc())

    medallion.df = (
        spark.read.table(SOURCE_TABLES["global_us_pkg_xref"])
        .withColumns(col_map)
        .select(*col_map.keys())
        .withColumn("rn", F.row_number().over(wind))
        .filter("rn = 1")
        .drop("rn")
        .filter(" and ".join([f"{col} is not NULL" for col in pks]))  # filters records where any of the pks is null
    )
except Exception as e:
    logger.error(f"Error processing global_package_xref table reads/joins. Error message: {e}")
    raise

In [0]:
# write gold table
try:
    medallion.write_to_delta(load_type="overwrite", layer="gold")
except Exception as e:
    logger.error(f"Error writing global_package_xref table to gold layer. Error message: {e}")
    raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()
except Exception as e:
    logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    logger.warning("Skipping comparison of legacy and modern because the required config is missing or invalid.")